# 00 — Setup and data check

Validate the self-contained `For Reviewer/` package before regenerating figures.

## Data availability

Most inputs are in `For_Reviewer/source_data/`. Two oversized tables (**Fig 2f–g, Fig 3**) are on
Zenodo — run `python setup/download_source_data.py` before those notebooks.

- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.
- No paths outside `For_Reviewer/` are used after packaging.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

In [ ]:

import pandas as pd
man = pd.read_csv(paths.SOURCE / "manifest.csv")
display(Markdown(f"**Manifest:** {len(man)} files, {man['bytes'].sum()/1e6:.1f} MB"))
display(man[["filename","bytes","panels"]].head(40))

ZENODO_LARGE = [
    "known_drug_rank_crispr_cancer_driver_role.csv",
    "docking_scores_fig2fg.csv",
]
missing_large = [r for r in ZENODO_LARGE if not (paths.SOURCE / r).exists()]
if missing_large:
    raise FileNotFoundError(
        "Missing Zenodo large tables (required for Fig 2f–g / Fig 3):\n  "
        + "\n  ".join(missing_large)
        + "\n\nRun:  python setup/download_source_data.py\n"
        + "Zenodo: https://doi.org/10.5281/zenodo.21615191"
    )

required = [
    "TableS2_Benchmarking_LinkD.xlsx",
    "drug_selectivity_metrics.csv",
    "Propranolol_growth.csv",
    "vct/propranolol/results_HR.csv",
    "adrenergic_selectivity_fig5.csv",
]
rows = []
for r in required:
    p = paths.SOURCE / r
    rows.append({"file": r, "present": p.exists(), "MB": round(p.stat().st_size/1e6, 2) if p.exists() else None})
ready = pd.DataFrame(rows)
display(ready)
assert ready["present"].all(), "Missing required source files — re-run setup/copy_and_extract_data.py"
print("All required files present (including Zenodo large tables).")